<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import Libraries

In [1]:
! pip install ultralytics easyocr

  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 4.6 MB/s eta 0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
# from google.colab.patches import cv2_imshow
import os
import pandas as pd

# Set model and path

In [11]:
objects_model = YOLO('TextDetection.pt') # set yolo model for object detection
card_model = YOLO('CardDetection.pt') # set yolo model for card detection
img_path = './test_image_phase1/2.jpg' # set image path
ocr = easyocr.Reader(['fa']) # set persian ocr reader

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [12]:
img_path = './test_image_phase1/2.jpg'

# Crop Card

In [13]:
def crop_card(path):
  img = cv2.imread(path) # read image

  results = card_model(img) # set card model for image

  for result in results:
    boxes = result.boxes
    for box in boxes:
      xyxy = box.xyxy[0]
      x1, y1, x2, y2 = map(int, xyxy.tolist()) # convert to int and set x and y

      # crop image and show that
      cropped_img = img[y1:y2, x1:x2]
      # cv2_imshow(cropped_img)
      return cropped_img

# Make crop image

In [14]:
crop = crop_card(img_path)


0: 288x640 1 Card, 226.1ms
Speed: 16.1ms preprocess, 226.1ms inference, 20.5ms postprocess per image at shape (1, 3, 288, 640)


# Make function for crop and read text parts

In [15]:
def process_img(img):
  ignore_class = ['FatherName', 'LastName', 'Name']
  data = {}
  results = objects_model(img) # set model for image
  for result in results:
    boxes = result.boxes
    for box in boxes:
      xyxy = box.xyxy[0]
      x1, y1, x2, y2 = map(int, xyxy.tolist()) # convert to int and set x and y

      label = result.names[int(box.cls)] # get class name

      if label in ignore_class: # ignore labels in ignore class
          continue

      # crop image and show that
      cropped_img = img[(y1 + 7):(y2 + 7), (x1 + 7):(x2 + 7)]
      # cv2_imshow(cropped_img)
      # set ocr and read text on the cropped image
      ocr_result = ocr.readtext(cropped_img)

      # OCR result
      for (bbox, text, conf) in ocr_result:
        print(f"Text: {text}, Confidence: {conf}")
        data[label] = text

  return data
          # draw box
        # cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
      # cv2_imshow(img)

# Call function and use it

In [16]:
def detect(folder):
  detected = []
  for img in os.listdir(folder):
    print(img)
    if img.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.heic')):
      img_path = os.path.join(folder, img)
      data = {'image_id': '', 'national_id': '', 'birth_year': '',
              'birth_month': '', 'birth_day': '', 'expiry_year': '',
              'expiry_month': '', 'expiry_day': ''}
      data['image_id'] = img.split('.')[0]
      card = crop_card(img_path)
      extracted_data = process_img(card)
      for key, value in extracted_data.items():
        value = value.replace(' ', '')
        if key == 'Expire':
          try:
            y, m, d = value.split('/')
            data['expiry_year'] = int(y)
            data['expiry_month'] = int(m)
            data['expiry_day'] = int(d)
          except:
            pass
        if key == 'Birth':
          try:
            y, m, d = value.split('/')
            data['expiry_year'] = int(y)
            data['expiry_month'] = int(m)
            data['expiry_day'] = int(d)
          except:
            pass
        if key == 'National':
          data['national_id'] = value
      detected.append(data)
  pd.DataFrame(detected).to_csv('image_phase1.csv', index=False, encoding='utf-8')


In [17]:
detect('./test_image_phase1')

1.heic
WARNING  'source' is missing. Using 'source=C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets'.

image 1/2 C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets\bus.jpg: 640x480 (no detections), 216.9ms
image 2/2 C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets\zidane.jpg: 384x640 (no detections), 208.9ms
Speed: 18.5ms preprocess, 212.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)
WARNING  'source' is missing. Using 'source=C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets'.

image 1/2 C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets\bus.jpg: 640x480 (no detections), 150.8ms
image 2/2 C:\Users\Radin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\assets\zidane.jpg: 384x640 (no detections), 130.5ms
Speed: 3.5ms preprocess, 140.6ms i